# Лабораторная работа № 1. Массивы NumPy и подготовка данных

Кейс: аудит данных интернет-магазина. В обязательных заданиях нет циклов `for`/`while`, Pandas и ручного перебора - только векторизованные операции NumPy и булевы маски. После кода каждого задания приведены аналитические выводы.

In [1]:
import numpy as np

np.set_printoptions(suppress=True, linewidth=120)


def section(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

## Исходные данные

In [2]:
order_id = np.array([1001, 1002, 1003, 1004, 1005, 1006])
quantity = np.array([2, 3, 0, 5, -1, 4])
unit_price = np.array([1500, 2000, 1200, 3000, 2500, np.nan])
discount = np.array([0.10, 0.05, 0.00, 0.15, 0.10, 0.20])

## Задание 1. Проверка структуры

In [3]:
section("Задание 1. Проверка структуры")


def describe(name, arr):
    print(f"{name:<11} ndim={arr.ndim}  shape={arr.shape}  "
          f"size={arr.size}  dtype={arr.dtype}")


describe("order_id", order_id)
describe("quantity", quantity)
describe("unit_price", unit_price)
describe("discount", discount)

same_shape = (order_id.shape == quantity.shape == unit_price.shape == discount.shape)
ids_unique = np.unique(order_id).size == order_id.size
print(f"\nФорма у всех массивов одинакова:        {same_shape}")
print(f"order_id уникальны (ключ без дублей):    {ids_unique}")
print(f"order_id упорядочены по возрастанию:     {bool(np.all(np.diff(order_id) > 0))}")
print("Вывод: сопоставлять по индексам МОЖНО, но только потому, что совпали форма,\n"
      "уникальность ключа и порядок. Одной длины для этого недостаточно (см. задание 6).")
print("Замечание: unit_price - float64, потому что содержит NaN (в int64 NaN не бывает).")


Задание 1. Проверка структуры
order_id    ndim=1  shape=(6,)  size=6  dtype=int64
quantity    ndim=1  shape=(6,)  size=6  dtype=int64
unit_price  ndim=1  shape=(6,)  size=6  dtype=float64
discount    ndim=1  shape=(6,)  size=6  dtype=float64

Форма у всех массивов одинакова:        True
order_id уникальны (ключ без дублей):    True
order_id упорядочены по возрастанию:     True
Вывод: сопоставлять по индексам МОЖНО, но только потому, что совпали форма,
уникальность ключа и порядок. Одной длины для этого недостаточно (см. задание 6).
Замечание: unit_price - float64, потому что содержит NaN (в int64 NaN не бывает).


### Выводы: Структура данных

| Массив | ndim | shape | size | dtype |
|---|---|---|---|---|
| `order_id` | 1 | (6,) | 6 | int64 |
| `quantity` | 1 | (6,) | 6 | int64 |
| `unit_price` | 1 | (6,) | 6 | float64 |
| `discount` | 1 | (6,) | 6 | float64 |

`unit_price` имеет тип `float64` только потому, что содержит `NaN`: в целочисленном массиве пропуск
представить нельзя.

Сопоставлять элементы по индексам можно, так как выполнены три условия: одинаковая форма,
`order_id` уникальны и упорядочены. Одинаковой длины для этого недостаточно (см. раздел 5).

## Задание 2. Контроль качества

In [4]:
section("Задание 2. Контроль качества (отдельная маска на каждую проблему)")

bad_quantity = quantity <= 0
bad_price = ~np.isfinite(unit_price)        # NaN и +-inf
bad_discount = (discount < 0) | (discount > 1)
failed_check = bad_quantity | bad_price | bad_discount

print("order_id          :", order_id)
print("bad_quantity      :", bad_quantity)
print("bad_price (NaN)   :", bad_price)
print("bad_discount      :", bad_discount)
print("failed_check (ИЛИ):", failed_check)
print("\nЗаказы с количеством <= 0     :", order_id[bad_quantity])
print("Заказы с пропущенной ценой    :", order_id[bad_price])
print("Заказы со скидкой вне [0, 1]  :", order_id[bad_discount])
print("Не прошли первичную проверку  :", order_id[failed_check],
      f"({failed_check.sum()} из {order_id.size})")


Задание 2. Контроль качества (отдельная маска на каждую проблему)
order_id          : [1001 1002 1003 1004 1005 1006]
bad_quantity      : [False False  True False  True False]
bad_price (NaN)   : [False False False False False  True]
bad_discount      : [False False False False False False]
failed_check (ИЛИ): [False False  True False  True  True]

Заказы с количеством <= 0     : [1003 1005]
Заказы с пропущенной ценой    : [1006]
Заказы со скидкой вне [0, 1]  : []
Не прошли первичную проверку  : [1003 1005 1006] (3 из 6)


### Выводы: Контроль качества

| Проблема | Условие | Заказы |
|---|---|---|
| Количество ≤ 0 | `quantity <= 0` | 1003 (0), 1005 (−1) |
| Пропущенная цена | `~np.isfinite(unit_price)` | 1006 (NaN) |
| Скидка вне [0, 1] | `(discount < 0) \| (discount > 1)` | нет |
| **Не прошли проверку** | объединение масок | **1003, 1005, 1006 (3 из 6, 50%)** |

Для каждой проблемы построена своя маска, поэтому видно, какой именно дефект у какой записи.

## Задание 3. Формирование рабочего набора

In [5]:
section("Задание 3. Рабочий набор (одна общая маска для всех массивов)")

valid = (
    (quantity > 0)
    & np.isfinite(unit_price)
    & (unit_price >= 0)
    & (discount >= 0)
    & (discount <= 1)
)
print("valid:", valid)

order_id_clean = order_id[valid]
quantity_clean = quantity[valid]
unit_price_clean = unit_price[valid]
discount_clean = discount[valid]

print("order_id_clean  :", order_id_clean)
print("quantity_clean  :", quantity_clean)
print("unit_price_clean:", unit_price_clean)
print("discount_clean  :", discount_clean)
print("Проверка: valid == ~failed_check ->", bool(np.array_equal(valid, ~failed_check)))

print("\nПочему нельзя фильтровать каждый массив независимо:")
q_only = quantity[quantity > 0]
p_only = unit_price[np.isfinite(unit_price)]
d_only = discount[(discount >= 0) & (discount <= 1)]
print(f"  длины при независимой фильтрации: quantity={q_only.size}, "
      f"price={p_only.size}, discount={d_only.size}  (должны совпадать)")
print("  quantity (независимо):", q_only)
print("  unit_price (независимо):", p_only)
print("  Теперь 3-й элемент quantity (5, это заказ 1004) стоит рядом с ценой 1200\n"
      "  (это цена заказа 1003, который был отброшен): связь между записями разрушена,\n"
      "  и дальше все расчёты выручки были бы молча неверными.")


Задание 3. Рабочий набор (одна общая маска для всех массивов)
valid: [ True  True False  True False False]
order_id_clean  : [1001 1002 1004]
quantity_clean  : [2 3 5]
unit_price_clean: [1500. 2000. 3000.]
discount_clean  : [0.1  0.05 0.15]
Проверка: valid == ~failed_check -> True

Почему нельзя фильтровать каждый массив независимо:
  длины при независимой фильтрации: quantity=4, price=5, discount=6  (должны совпадать)
  quantity (независимо): [2 3 5 4]
  unit_price (независимо): [1500. 2000. 1200. 3000. 2500.]
  Теперь 3-й элемент quantity (5, это заказ 1004) стоит рядом с ценой 1200
  (это цена заказа 1003, который был отброшен): связь между записями разрушена,
  и дальше все расчёты выручки были бы молча неверными.


### Выводы: Рабочий набор

Одна общая маска `valid` (количество > 0, цена конечна и ≥ 0, скидка в [0, 1]) оставила заказы
**1001, 1002, 1004**. Она в точности равна `~failed_check`, то есть проверки согласованы.

**Почему нельзя фильтровать массивы независимо.** Каждый массив — это один столбец таблицы,
а строка (заказ) задаётся позицией. Если отфильтровать их по своим условиям, получатся разные длины
(`quantity` — 4, `unit_price` — 5, `discount` — 6), а равные длины были бы лишь случайностью.
В результате количество 5 (заказ 1004) оказывается рядом с ценой 1200 (заказ 1003), и выручка
считается неверно без какой-либо ошибки. Решение — одна булева маска, применяемая ко всем массивам сразу.

## Задание 4. Подготовка матрицы

In [6]:
section("Задание 4. Матрица валидных заказов")

features = np.column_stack((quantity_clean, unit_price_clean, discount_clean))
print(features)
print("shape:", features.shape, "| dtype:", features.dtype)
print("Строки = заказы (наблюдения), столбцы = признаки [quantity, unit_price, discount].")
print("Форма (n_orders, n_features) - стандартное соглашение для аналитики:\n"
      "агрегации по axis=0 дают статистику признака, по axis=1 - по заказу.")
print("Все столбцы приведены к float64 (общий тип матрицы); целое quantity стало вещественным.")


Задание 4. Матрица валидных заказов
[[   2.   1500.      0.1 ]
 [   3.   2000.      0.05]
 [   5.   3000.      0.15]]
shape: (3, 3) | dtype: float64
Строки = заказы (наблюдения), столбцы = признаки [quantity, unit_price, discount].
Форма (n_orders, n_features) - стандартное соглашение для аналитики:
агрегации по axis=0 дают статистику признака, по axis=1 - по заказу.
Все столбцы приведены к float64 (общий тип матрицы); целое quantity стало вещественным.


### Выводы: Матрица

`np.column_stack((quantity, price, discount))` для валидных заказов даёт форму **(3, 3)**: строки —
заказы, столбцы — признаки `[quantity, unit_price, discount]`. Так организовано потому, что:

- это стандартное соглашение «наблюдения × признаки»: `axis=0` даёт статистику по признаку,
  `axis=1` — по заказу;
- заказ остаётся целой строкой, и его нельзя «разорвать» при сортировке или фильтрации.

Побочный эффект: матрица имеет единый тип `float64`, поэтому целое `quantity` стало вещественным.

## Задание 5. Преобразование данных

In [7]:
section("Задание 5. Преобразование формы")

rng = np.random.default_rng(42)
sales_flat = rng.integers(10_000, 50_000, size=24)
print("Исходный вектор (24):", sales_flat)

weeks_by_cat = sales_flat.reshape(4, 6)      # 4 недели x 6 категорий
cat_by_weeks_reshape = sales_flat.reshape(6, 4)
cat_by_weeks_transpose = weeks_by_cat.T

print("\nreshape(4, 6) - недели x категории:\n", weeks_by_cat)
print("\nreshape(6, 4):\n", cat_by_weeks_reshape)
print("\nweeks_by_cat.T (6, 4) - категории x недели:\n", cat_by_weeks_transpose)
print("\nreshape(6, 4) совпадает с транспонированием?",
      bool(np.array_equal(cat_by_weeks_reshape, cat_by_weeks_transpose)))
print("Размеры согласованы: 4*6 == 6*4 == size ->", 4 * 6 == 6 * 4 == sales_flat.size)
print("Вывод: reshape лишь перечитывает те же 24 числа по строкам с новой шириной,\n"
      "и смысл строк/столбцов он НЕ переносит. Чтобы получить 'категории x недели',\n"
      "нужно транспонирование (.T), а не reshape(6, 4).")
print("Пример: weeks_by_cat[1, 0] (неделя 2, кат. 1) =", weeks_by_cat[1, 0],
      "| reshape(6,4)[1, 0] =", cat_by_weeks_reshape[1, 0],
      "| .T[0, 1] (кат. 1, неделя 2) =", cat_by_weeks_transpose[0, 1])


Задание 5. Преобразование формы
Исходный вектор (24): [13570 40958 36182 27555 27320 44343 13437 37894 18058 13767 31059 49024 39430 40445 38699 41442 30529 15124 43589
 28015 30014 24831 17301 47070]

reshape(4, 6) - недели x категории:
 [[13570 40958 36182 27555 27320 44343]
 [13437 37894 18058 13767 31059 49024]
 [39430 40445 38699 41442 30529 15124]
 [43589 28015 30014 24831 17301 47070]]

reshape(6, 4):
 [[13570 40958 36182 27555]
 [27320 44343 13437 37894]
 [18058 13767 31059 49024]
 [39430 40445 38699 41442]
 [30529 15124 43589 28015]
 [30014 24831 17301 47070]]

weeks_by_cat.T (6, 4) - категории x недели:
 [[13570 13437 39430 43589]
 [40958 37894 40445 28015]
 [36182 18058 38699 30014]
 [27555 13767 41442 24831]
 [27320 31059 30529 17301]
 [44343 49024 15124 47070]]

reshape(6, 4) совпадает с транспонированием? False
Размеры согласованы: 4*6 == 6*4 == size -> True
Вывод: reshape лишь перечитывает те же 24 числа по строкам с новой шириной,
и смысл строк/столбцов он НЕ переносит

## Задание 6. Проверка источников

In [8]:
section("Задание 6. Проверка источников")

# Файл заказов: id 1001..1006. В файле цен записи 1003 нет, зато есть лишняя 1007.
ids_orders_file = np.array([1001, 1002, 1003, 1004, 1005, 1006])
ids_prices_file = np.array([1001, 1002, 1004, 1005, 1006, 1007])
prices_file = np.array([1500.0, 2000.0, 3000.0, 2500.0, 1800.0, 2200.0])

print("Случай А: в файле цен просто не хватает строки (5 значений вместо 6)")
prices_short = prices_file[:5]
qty_file = np.array([2, 3, 0, 5, -1, 4])
print("  shape заказов:", qty_file.shape, "| shape цен:", prices_short.shape)
try:
    _ = qty_file * prices_short
except ValueError as err:
    print("  Умножение упало с ошибкой:", err)

print("\nСлучай Б: длины равны (6 и 6), но набор записей разный")
print("  shape совпадают:", ids_orders_file.shape == ids_prices_file.shape)
print("  id заказов:", ids_orders_file)
print("  id цен    :", ids_prices_file)
print("  Совпадение по позиции:", ids_orders_file == ids_prices_file)
print("  Одинаковые наборы id :",
      bool(np.array_equal(np.sort(ids_orders_file), np.sort(ids_prices_file))))
print("  Нет цены для заказов :", np.setdiff1d(ids_orders_file, ids_prices_file))
print("  Лишние цены          :", np.setdiff1d(ids_prices_file, ids_orders_file))
print("  Уникальность id (заказы/цены):",
      np.unique(ids_orders_file).size == ids_orders_file.size,
      np.unique(ids_prices_file).size == ids_prices_file.size)
wrong_qty_price = qty_file * prices_file          # молча считается, но неверно!
print("  Наивное qty * price по позициям (ОШИБОЧНО):", wrong_qty_price)

print("\nКорректное сопоставление по ключу (searchsorted):")
common = np.intersect1d(ids_orders_file, ids_prices_file)
order_sort = np.argsort(ids_prices_file)
pos = np.searchsorted(ids_prices_file[order_sort], common)
matched_prices = prices_file[order_sort[pos]]
matched_qty = qty_file[np.isin(ids_orders_file, common)]
print("  общие id      :", common)
print("  qty           :", matched_qty)
print("  цены          :", matched_prices)
print("  qty * price   :", matched_qty * matched_prices)
print("\nПеред сопоставлением нужно проверить: 1) shape; 2) уникальность ключа;\n"
      "3) совпадение множеств ключей (setdiff1d в обе стороны); 4) порядок строк;\n"
      "5) типы и NaN/inf. Равная длина гарантирует лишь возможность поэлементной\n"
      "операции, но не то, что i-я запись в одном файле - это тот же заказ, что в другом.")


Задание 6. Проверка источников
Случай А: в файле цен просто не хватает строки (5 значений вместо 6)
  shape заказов: (6,) | shape цен: (5,)
  Умножение упало с ошибкой: operands could not be broadcast together with shapes (6,) (5,) 

Случай Б: длины равны (6 и 6), но набор записей разный
  shape совпадают: True
  id заказов: [1001 1002 1003 1004 1005 1006]
  id цен    : [1001 1002 1004 1005 1006 1007]
  Совпадение по позиции: [ True  True False False False False]
  Одинаковые наборы id : False
  Нет цены для заказов : [1003]
  Лишние цены          : [1007]
  Уникальность id (заказы/цены): True True
  Наивное qty * price по позициям (ОШИБОЧНО): [ 3000.  6000.     0. 12500. -1800.  8800.]

Корректное сопоставление по ключу (searchsorted):
  общие id      : [1001 1002 1004 1005 1006]
  qty           : [ 2  3  5 -1  4]
  цены          : [1500. 2000. 3000. 2500. 1800.]
  qty * price   : [ 3000.  6000. 15000. -2500.  7200.]

Перед сопоставлением нужно проверить: 1) shape; 2) уникальность кл

### Выводы: Преобразование формы и проверка источников

**Форма.** 24 значения продаж: `reshape(4, 6)` — недели × категории, `reshape(6, 4)` — другая матрица.
Она **не равна** транспонированию (`np.array_equal(...)` → `False`): `reshape` просто перечитывает те же
24 числа по строкам с другой шириной и не переносит смысл осей. Например, значение «неделя 2, категория 1»
(13 437) в `reshape(6, 4)` окажется в другой ячейке, а в `.T` — на месте `[0, 1]`. Если нужны
«категории × недели», нужно `.T`, а не `reshape`.

**Источники.** Смоделированы два случая:

- *А.* В файле цен 5 значений вместо 6 — NumPy сразу падает с `ValueError` (broadcast), то есть ошибка видна.
- *Б.* В обоих файлах по 6 записей, но в ценах нет заказа 1003 и есть лишний 1007. Операция проходит без ошибки и
  даёт неверные числа (например, для заказа 1004 при наивном `qty * price` получилось бы 12 500 вместо верных 15 000, потому что подставилась цена соседнего заказа).

Это и есть ответ на вопрос задания: **равное число элементов не гарантирует, что i-я запись в одном файле
описывает тот же объект, что i-я запись в другом**. Перед сопоставлением необходимо проверить:

1. совпадение `shape`;
2. уникальность ключа (`np.unique(...).size == size`);
3. совпадение множеств ключей в обе стороны (`np.setdiff1d`);
4. порядок строк (или сопоставлять по ключу через `np.searchsorted`/`np.isin`, а не по позиции);
5. типы и наличие `NaN`/`inf`.

## Задание 7. Подготовка к аналитике

In [9]:
section("Задание 7. Матрица продаж 4 региона x 6 недель")

n_regions, n_weeks = 4, 6
sales = rng.integers(80_000, 250_000, size=(n_regions, n_weeks))
print(sales)

has_nan = np.isnan(sales).any()
has_inf = np.isinf(sales).any()
has_negative = (sales < 0).any()
shape_ok = sales.shape == (n_regions, n_weeks)
print("\nЕсть NaN:        ", bool(has_nan))
print("Есть inf:        ", bool(has_inf))
print("Есть отрицательные:", bool(has_negative))
print("Форма == (регионы, недели):", shape_ok, sales.shape)
print("Матрица готова к аналитике:", bool(not has_nan and not has_inf and not has_negative and shape_ok))

print("\nКонтрольный прогон на намеренно испорченной копии:")
broken = sales.astype(float)
broken[1, 2] = np.nan
broken[0, 0] = np.inf
broken[3, 5] = -1000.0
print("  NaN:", np.argwhere(np.isnan(broken)).tolist(),
      "| inf:", np.argwhere(np.isinf(broken)).tolist(),
      "| <0:", np.argwhere(broken < 0).tolist())
print("  Проверки находят все три дефекта и точные позиции (регион, неделя).")


Задание 7. Матрица продаж 4 региона x 6 недель
[[212866 189457 148410 219869 172722 155380]
 [156578 118630  95663 174279 230941  90848]
 [225909 220697 127049 187382 108088 208874]
 [199088 140269  91546 245018 155766 231830]]

Есть NaN:         False
Есть inf:         False
Есть отрицательные: False
Форма == (регионы, недели): True (4, 6)
Матрица готова к аналитике: True

Контрольный прогон на намеренно испорченной копии:
  NaN: [[1, 2]] | inf: [[0, 0]] | <0: [[3, 5]]
  Проверки находят все три дефекта и точные позиции (регион, неделя).


### Выводы: Подготовка к аналитике

Матрица продаж 4 региона × 6 недель: нет `NaN`, нет `inf`, нет отрицательных значений, форма `(4, 6)` соответствует
числу регионов и недель, поэтому её можно агрегировать. Контрольный прогон на намеренно испорченной копии
находит все три дефекта вместе с позициями (регион, неделя), то есть сами проверки рабочие.

## Задание 8. Аналитическое заключение

In [10]:
section("Задание 8. Сводка проблем")
print(f"Заказов всего: {order_id.size}, годных: {valid.sum()} ({valid.mean():.0%}), "
      f"отбраковано: {failed_check.sum()} ({failed_check.mean():.0%})")
print("Критические : NaN в unit_price (1006); quantity < 0 (1005)")
print("Спорная     : quantity == 0 (1003) - вероятно отмена/пустая позиция")
print("Не выявлено : скидки вне [0, 1], дубли order_id")


Задание 8. Сводка проблем
Заказов всего: 6, годных: 3 (50%), отбраковано: 3 (50%)
Критические : NaN в unit_price (1006); quantity < 0 (1005)
Спорная     : quantity == 0 (1003) - вероятно отмена/пустая позиция
Не выявлено : скидки вне [0, 1], дубли order_id


### Выводы: Аналитическое заключение

| Проблема | Заказ | Класс | Последствия для анализа |
|---|---|---|---|
| Цена `NaN` | 1006 | **Критическая** | Любая сумма/среднее по выручке станет `NaN` (`np.sum` заразит весь результат); при замене нулём выручка занижается без видимой ошибки. Заказ нужно исключить или восстановить цену из первоисточника. |
| Количество < 0 | 1005 | **Критическая** | Если это возврат, его нужно учитывать отдельно, иначе он вычтет выручку из итога; если ошибка ввода — исказит выручку и средний чек. Требует уточнения у источника. |
| Количество = 0 | 1003 | Некритическая (требует пояснения) | Выручка заказа равна нулю, сумму не искажает, но занижает средний чек и число «реальных» заказов. Вероятно, отменённый или пустой заказ — стоит выяснить, почему запись попала в выгрузку. |
| Скидки вне [0, 1] | — | Не обнаружены | Диапазон соблюдён, проблем нет. |
| Дубли `order_id` | — | Не обнаружены | Ключ уникален. |
| Тип `float64` из-за `NaN` | `unit_price` | Некритическая | Влияет на тип матрицы: целые становятся вещественными; после очистки можно привести обратно. |

**Итог.** Из 6 заказов к финансовым расчётам пригодны 3 (50%). Исходный набор для финансовых расчётов
**использовать нельзя** без очистки: две из трёх проблем критические, и оба дефекта дали бы неверные суммы без явной ошибки.
Очищенный набор (1001, 1002, 1004) допустим, но слишком мал: перед выводами по бизнесу нужно запросить корректные данные
по отбракованным заказам, а не просто отбросить 50% выборки.

## Дополнительное задание: универсальная функция аудита

In [11]:
section("Дополнительное задание. audit_array()")


def audit_array(arr):
    """Паспорт качества числового массива (без циклов, только векторные операции)."""
    arr = np.asarray(arr)
    if not np.issubdtype(arr.dtype, np.number):
        raise TypeError(f"Ожидался числовой массив, получен dtype={arr.dtype}")
    is_nan = np.isnan(arr)
    is_inf = np.isinf(arr)
    is_neg = arr < 0                       # NaN < 0 -> False, NaN в негативы не попадает
    return {
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "n_nan": int(is_nan.sum()),
        "n_inf": int(is_inf.sum()),
        "n_negative": int(is_neg.sum()),
        "nan_positions": np.argwhere(is_nan).tolist(),
        "inf_positions": np.argwhere(is_inf).tolist(),
        "negative_positions": np.argwhere(is_neg).tolist(),
    }


print("quantity   :", audit_array(quantity))
print("unit_price :", audit_array(unit_price))
print("discount   :", audit_array(discount))
print("sales (4x6):", audit_array(sales))
print("broken     :", audit_array(broken))
try:
    audit_array(np.array(["a", "b"]))
except TypeError as err:
    print("строковый массив:", err)


Дополнительное задание. audit_array()
quantity   : {'shape': (6,), 'dtype': 'int64', 'n_nan': 0, 'n_inf': 0, 'n_negative': 1, 'nan_positions': [], 'inf_positions': [], 'negative_positions': [[4]]}
unit_price : {'shape': (6,), 'dtype': 'float64', 'n_nan': 1, 'n_inf': 0, 'n_negative': 0, 'nan_positions': [[5]], 'inf_positions': [], 'negative_positions': []}
discount   : {'shape': (6,), 'dtype': 'float64', 'n_nan': 0, 'n_inf': 0, 'n_negative': 0, 'nan_positions': [], 'inf_positions': [], 'negative_positions': []}
sales (4x6): {'shape': (4, 6), 'dtype': 'int64', 'n_nan': 0, 'n_inf': 0, 'n_negative': 0, 'nan_positions': [], 'inf_positions': [], 'negative_positions': []}
broken     : {'shape': (4, 6), 'dtype': 'float64', 'n_nan': 1, 'n_inf': 1, 'n_negative': 1, 'nan_positions': [[1, 2]], 'inf_positions': [[0, 0]], 'negative_positions': [[3, 5]]}
строковый массив: Ожидался числовой массив, получен dtype=<U1


### Выводы: Дополнительное задание

Функция `audit_array(arr)` возвращает форму, тип, число и позиции `NaN`, `inf` и отрицательных элементов с помощью
векторных операций. Для нечисловых массивов выбрасывает `TypeError` (поскольку `np.isnan` на строках не работает).
Проверена на `quantity`, `unit_price`, `discount`, матрице продаж и намеренно испорченной матрице. Отрицательные
значения считает отдельно от `NaN`, так как `NaN < 0` даёт `False`.